# 🚀 Фабрика нейросетей & Асинхронный Walk-Forward рой PRL/USDT (Google Colab / GPU T4)
Весь тяжёлый код архитектур и обучения вынесен в модульные скрипты  и .
Управляйте параметрами обучения напрямую через аргументы в ячейках ниже.

In [1]:
!nvidia-smi
import os, sys, importlib
if os.path.exists("/content"):
    !mkdir -p _tools
    !curl -sL https://raw.githubusercontent.com/BulletproofBoris/crypto_bot/main/_tools/colab_models.py -o _tools/colab_models.py
    !curl -sL https://raw.githubusercontent.com/BulletproofBoris/crypto_bot/main/_tools/colab_neural_trainer.py -o _tools/colab_neural_trainer.py
sys.path.insert(0, os.path.abspath("."))
import _tools.colab_models, _tools.colab_neural_trainer
importlib.reload(_tools.colab_models)
importlib.reload(_tools.colab_neural_trainer)
from _tools.colab_neural_trainer import run_colab_pipeline
print("✅ Свежие модули _tools.colab_models и _tools.colab_neural_trainer загружены!")

Sat Oct 24 06:10:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# 2. Конфигурация и запуск Walk-Forward обучения нейросетей (редактируемые параметры)
# Асинхронная сетка окон: H=25 -> 2 окна (375k), H=12 -> 5 окон (210k), H=5 -> 12 окон (75k)
FILE_ID = "1XnalYuDXB8ywwPw9siVNWlFKH1bbjT7O"              # Свежий датасет от 23.10 (4 окна H=25)
HORIZON = 25                                               # Горизонт H (сек): 5, 12 или 25
WINDOWS = "0"                                              # Список окон: "all" или "0,1"
#ARCHS = "conv1d+gru,causal_cnn,attention,mlp_bottleneck"
ARCHS = "attention,mlp_bottleneck"                         # Архитектуры через запятую
RUNS_PER_ARCH = 15                                         # Число независимых ранов на архитектуру
MAX_EPOCHS = 10                                            # Макс. эпох на один ран
BATCH_SIZE = 2048                                          # Размер батча на GPU T4
LR = 5e-3                                                  # Скорость обучения AdamW

df_leaderboard = run_colab_pipeline(
    file_id=FILE_ID,
    horizon=HORIZON, windows=WINDOWS, archs=ARCHS,
    runs_per_arch=RUNS_PER_ARCH, max_epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE, lr=LR
)

In [3]:
# 3. Выгрузка готового архива моделей (авто-упаковка при прерванной 2-й ячейке)
import os, shutil
HORIZON = globals().get("HORIZON", 25)
!pkill -9 -f "colab_neural_trainer" 2>/dev/null || true
PACK_DIR = f"prl_ensemble_pack_h{HORIZON}s"
ZIP_NAME = f"{PACK_DIR}.zip"
if os.path.exists(PACK_DIR):
    shutil.make_archive(PACK_DIR, "zip", root_dir=".", base_dir=PACK_DIR)
if os.path.exists(ZIP_NAME):
    print(f"📦 Архив найден: {ZIP_NAME} ({os.path.getsize(ZIP_NAME) / (1024*1024):.2f} MB)")
    !curl -s --max-time 60 -F "file=@{ZIP_NAME}" https://x0.at/ | head -c 200 && echo " (x0.at)"
    !curl -s --max-time 60 -F "file=@{ZIP_NAME}" https://store1.gofile.io/uploadFile | head -c 400 && echo " (gofile)"
    !curl -s --max-time 60 -H "Accept: application/json" -T "{ZIP_NAME}" https://filebin.net/prl_h{HORIZON}s_pack/{ZIP_NAME} | head -c 300 && echo " (filebin)"

^C
📦 Архив найден: prl_ensemble_pack_h25s.zip (4.13 MB)
https://x0.at/v1Oo.zip
 (x0.at)
{"data":{"code":"eXRAyP2v","createTime":1792673906,"downloadPage":"https://gofile.io/d/ZRqhrLba","guestToken":"JI16CGid1XadyevFmZOTme8STBzkgTIr","id":"9c5f9a67-4498-4e2c-80d2-0e4e3096da03","md5":"c2e0f03defea76f70349823dfa677c3d","mimetype":"application/zip","modTime":1792673906,"name":"prl_ensemble_pack_h25s.zip","parentFolder":"de73ae7e-be9a-4fab-8400-ada7563a2bc6","parentFolderCode":"ZRqhrLba", (gofile)
{
    "bin": {
        "id": "prl_h25s_pack",
        "readonly": false,
        "bytes": 5649506,
        "bytes_readable": "5.6 MB",
        "files": 1,
        "updated_at": "2026-10-22T12:58:38.294859Z",
        "updated_at_relative": "now",
        "created_at": "2026-10-21T12:17:32.137515Z",
  (filebin)
